In [1]:
#| default_exp rest

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [3]:
model_path = 'pelevin'

In [4]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
import torch
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(model_path,torch_dtype=torch.bfloat16) #.half()
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

In [5]:
sum(p.numel() for p in model.parameters())

774030080

In [6]:
#| export
import threading
lock = threading.RLock()

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    with lock:
        return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [9]:
%%time
get_sample(' - ты кто?', 50, 4, False)

CPU times: user 13.3 s, sys: 279 ms, total: 13.5 s
Wall time: 1.14 s


[' Наркота? А-а, знаю. Тульский налет. Давай снимай курточку, мужик. Полезай в ведро. А то Вася сейчас вернется. Или Ольга. Он на самом деле такой крутой?',
 ' Дворник? Химик? Я ведь ничего про тебя не знаю. А если ты вор, мне все равно, вор ты или нет. Вот и все. Бери паспорт и проваливай. И денег не жалко. У меня их и так достаточно.',
 ' - тихо спросил я, глядя в его лицо. Фаза проворчал что-то невнятное, поднялся на ноги и быстро пошел к тому месту, где мы сидели, прямо на людей. Через секунду его уже не было видно.',
 ' Даже не знаю, как тебя назвать. Как пса, что ли? Если тебя звать Цербером, должен соответствовать.']